# NB5e — Chronos-2 at every PVSetFormer issue time (Colab GPU)

D70. Runs `tools/fm_infer.py` with A6a's part-2d setting (Chronos-2, 1,344 daylight steps, covariates on) on `results/nb5e_inputs/`, one part per validation event, into `results/nb5e_runs/`. Resumable: a part whose output exists for the current inputs is skipped. Development data only.

**On Colab:** open this notebook from the Drive folder `pv-cold-start-nb5e` (made by `python tools/make_drive_mirror.py --set nb5e`), choose a GPU runtime, run all cells. The last cell unmounts Drive; then Ctrl+S. Back on the local machine: `python tools/fetch_drive_runs.py --set nb5e`.

In [ ]:
import os, sys, json, hashlib, subprocess, time
from pathlib import Path
ON_COLAB = 'google.colab' in sys.modules
DRIVE_REPO = '/content/drive/MyDrive/pv-cold-start-nb5e'
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT, PY, DEVICE = Path(DRIVE_REPO), sys.executable, 'cuda'
    subprocess.run([PY, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements-fm.txt')], check=True)
else:
    ROOT = Path.cwd().resolve(); ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
    PY = str(ROOT / '.venv_fm' / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')); DEVICE = 'cpu'
INP, RUNS = ROOT / 'results' / 'nb5e_inputs', ROOT / 'results' / 'nb5e_runs'
man = json.loads((INP / 'inputs_manifest.json').read_text())
for f, h in man['sha256'].items():
    assert hashlib.sha256((INP / f).read_bytes()).hexdigest() == h, f'{f} differs from its manifest'
EVENTS = ['SQ Apartment1-12', 'SQ Apartment13-24', 'SQ Apartment25-36', 'SQ Apartment37-48', 'SQ567', 'UG Hall2 2F', 'UG Hall2 RF', 'Zone A1', 'Zone A2 1', 'Zone A4']
MODEL, CONTEXT, COV = 'chronos2', 1344, 'full'
print('ROOT', ROOT, '| device', DEVICE, '| events', len(EVENTS), '| tasks', man['n_tasks'])

In [ ]:
log = []
for i, v in enumerate(EVENTS):
    out = RUNS / f'{MODEL}_c{CONTEXT}_{COV}__e{i:02d}.parquet'
    js = out.with_suffix('.json')
    if out.exists() and js.exists() and json.loads(js.read_text())['inputs_sha256'] == man['sha256']:
        log.append({'part': out.stem, 'event': v, 'status': 'done before'})
        continue
    t1 = time.time()
    r = subprocess.run([PY, str(ROOT / 'tools' / 'fm_infer.py'), '--model', MODEL, '--context', str(CONTEXT),
                        '--cov', COV, '--inputs', str(INP), '--out', str(out), '--device', DEVICE,
                        '--events', v], cwd=str(ROOT))
    assert r.returncode == 0, f'{out.stem} ({v}) failed'
    log.append({'part': out.stem, 'event': v, 'status': 'run', 'runtime_s': round(time.time() - t1, 1),
                'n_rows': json.loads(js.read_text())['n_rows']})
    print(log[-1], flush=True)
print(log)

In [ ]:
if ON_COLAB:
    drive.flush_and_unmount()